# Lesson 1 — Gradient checking & writing your own operations

`Value.backward()` computes gradients with the chain rule. But how do we *know* those gradients are right?
In this lesson you will build the tool every autograd engine relies on — a **numerical gradient checker** —
and then use it to verify new operations you write yourself.

**You will learn to**
1. Estimate a derivative numerically, and see why the step size `h` matters
2. Build `numerical_grad` and `gradcheck`
3. Write the backward pass for `exp`, `log`, `tanh` and `sigmoid`
4. Break the engine on purpose to understand why gradients are *accumulated* with `+=`

Cells tagged **Exercise** contain `raise NotImplementedError` for you to replace. Each exercise is followed by a
check cell and a collapsed **Solution** cell. Running the notebook top-to-bottom always works, because the solution
cells overwrite your attempt — so try the exercise and run its check *before* you expand the solution.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))  # so `import nanograd` works from the lessons/ folder without installing

def check(name, test):
    """ run a test function; print a tick, a cross, or a reminder if not implemented yet """
    try:
        ok = test()
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet")
        return
    except Exception as e:
        print(f"❌ {name}: raised {type(e).__name__}: {e}")
        return
    print(f"{'✅' if ok else '❌'} {name}")

In [ ]:
import math
from nanograd.engine import Value

## Part 1 — The derivative is a slope

The derivative of `f` at `x` asks: *if I nudge `x` a tiny bit, how much does `f` move?*

$$f'(x) \approx \frac{f(x+h) - f(x-h)}{2h}$$

This *central difference* needs no calculus at all — only two evaluations of `f`. Let's try it on
$f(x) = x^3 - 2x$, whose true derivative is $3x^2 - 2$.

In [ ]:
f = lambda x: x**3 - 2*x
df = lambda x: 3*x**2 - 2

x = 1.5
print(f"true derivative: {df(x)}")
print(f"{'h':>8} {'forward diff error':>20} {'central diff error':>20}")
for h in [1e-1, 1e-2, 1e-4, 1e-6, 1e-8, 1e-10, 1e-12]:
    forward = (f(x + h) - f(x)) / h
    central = (f(x + h) - f(x - h)) / (2 * h)
    print(f"{h:>8.0e} {abs(forward - df(x)):>20.2e} {abs(central - df(x)):>20.2e}")

**Read the table carefully — it holds two lessons:**

* The central difference is far more accurate than the one-sided (forward) one for the same `h`. Its error shrinks like $h^2$ instead of $h$.
* Smaller `h` is *not* always better. Below ~`1e-6` the error grows again: `f(x+h)` and `f(x-h)` become nearly equal
  floating-point numbers, and subtracting them throws away most of the significant digits (*catastrophic cancellation*).
  This is why `h = 1e-6` is the usual choice for float64.

### Exercise 1.1 — `numerical_grad`

Our functions take **several** inputs. Write `numerical_grad(f, xs, h)` where

* `xs` is a list of floats,
* `f` takes a list of `Value`s and returns a single `Value`,
* the result is a list with the estimated $\partial f / \partial x_i$ for every input.

For each `i`, build two input lists where only `xs[i]` is nudged by `+h` / `-h`, run `f` on both, and take the central difference.

In [ ]:
def numerical_grad(f, xs, h=1e-6):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
# f(a, b) = a*b + a^2  ->  df/da = b + 2a = 7,  df/db = a = 2   at (a, b) = (2, 3)
g = lambda v: v[0] * v[1] + v[0]**2
check("numerical_grad", lambda: all(abs(n - e) < 1e-6 for n, e in zip(numerical_grad(g, [2.0, 3.0]), [7.0, 2.0])))

<details><summary><b>Solution</b> (try the exercise first!)</summary>The next cell is collapsed — click it to expand.</details>

In [ ]:
def numerical_grad(f, xs, h=1e-6):
    grads = []
    for i in range(len(xs)):
        plus  = [Value(x + h if j == i else x) for j, x in enumerate(xs)]
        minus = [Value(x - h if j == i else x) for j, x in enumerate(xs)]
        grads.append((f(plus).data - f(minus).data) / (2 * h))
    return grads

check("numerical_grad", lambda: all(abs(n - e) < 1e-6 for n, e in zip(numerical_grad(g, [2.0, 3.0]), [7.0, 2.0])))

### Exercise 1.2 — `gradcheck`

Now compare against backprop. Write `gradcheck(f, xs, tol)` that

1. wraps `xs` in `Value`s, runs `f`, calls `.backward()` and collects each input's `.grad` (the *analytic* gradient),
2. computes the *numerical* gradient with your `numerical_grad`,
3. returns `True` if every pair agrees within `tol` using the relative error below.

We use a *relative* error so that the check works for both tiny and huge gradients; the `max(1.0, ...)` keeps it from blowing up
when both gradients are close to zero.

In [ ]:
def rel_error(a, b):
    return abs(a - b) / max(1.0, abs(a), abs(b))

In [ ]:
def gradcheck(f, xs, tol=1e-5):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
def broken_square(v):
    """ v**2 with a deliberately wrong backward pass (forgot the factor 2) """
    out = Value(v.data**2, (v,), 'sq')
    def _backward():
        v.grad += v.data * out.grad
    out._backward = _backward
    return out

check("gradcheck accepts a correct gradient", lambda: gradcheck(lambda v: v[0] * v[1] + v[0]**3 / v[1], [1.5, -2.0]))
check("gradcheck rejects a wrong gradient", lambda: not gradcheck(lambda v: broken_square(v[0]), [1.5]))

<details><summary><b>Solution</b> (try the exercise first!)</summary>The next cell is collapsed — click it to expand.</details>

In [ ]:
def gradcheck(f, xs, tol=1e-5):
    vals = [Value(x) for x in xs]
    f(vals).backward()
    analytic = [v.grad for v in vals]
    numeric = numerical_grad(f, xs)
    return all(rel_error(a, n) < tol for a, n in zip(analytic, numeric))

check("gradcheck accepts a correct gradient", lambda: gradcheck(lambda v: v[0] * v[1] + v[0]**3 / v[1], [1.5, -2.0]))
check("gradcheck rejects a wrong gradient", lambda: not gradcheck(lambda v: broken_square(v[0]), [1.5]))

The library ships the same function in `nanograd.gradcheck`, with a `verbose` table that is handy for debugging.
Let's run it on the big expression from the README:

In [ ]:
from nanograd.gradcheck import gradcheck as lib_gradcheck

def readme_expr(v):
    a, b = v
    c = a + b
    d = a * b + b**3
    c += c + 1
    c += 1 + c + (-a)
    d += d * 2 + (b + a).relu()
    d += 3 * d + (b - a).relu()
    e = c - d
    f = e**2
    g = f / 2.0
    g += 10.0 / f
    return g

lib_gradcheck(readme_expr, [-4.0, 2.0], verbose=True)

## Part 2 — Writing new operations

Every operation in `engine.py` follows the same recipe:

```python
def op(self):
    out = Value(<forward value>, (self,), 'op')      # 1. compute the output, remember the inputs
    def _backward():
        self.grad += <local derivative> * out.grad   # 2. chain rule: local derivative x upstream gradient
    out._backward = _backward
    return out
```

The *local derivative* is the only part that changes from op to op. Below you write each op as a plain function
`op(v)` that takes a `Value`. (The finished versions live in `nanograd/engine.py` as `v.exp()`, `v.log()`, `v.tanh()`.)

### Exercise 2.1 — `exp`
$\frac{d}{dx} e^x = e^x$. *Hint: you have already computed $e^x$ in the forward pass — reuse `out.data`.*

In [ ]:
def exp(v):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
check("exp forward", lambda: abs(exp(Value(1.0)).data - math.e) < 1e-12)
check("exp backward", lambda: gradcheck(lambda v: exp(v[0] * v[1]), [0.5, -1.5]))

<details><summary><b>Solution</b> (try the exercise first!)</summary>The next cell is collapsed — click it to expand.</details>

In [ ]:
def exp(v):
    out = Value(math.exp(v.data), (v,), 'exp')
    def _backward():
        v.grad += out.data * out.grad
    out._backward = _backward
    return out

check("exp forward", lambda: abs(exp(Value(1.0)).data - math.e) < 1e-12)
check("exp backward", lambda: gradcheck(lambda v: exp(v[0] * v[1]), [0.5, -1.5]))

### Exercise 2.2 — `log`
$\frac{d}{dx} \ln x = \frac{1}{x}$. Only defined for $x > 0$.

In [ ]:
def log(v):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
check("log forward", lambda: abs(log(Value(math.e)).data - 1.0) < 1e-12)
check("log backward", lambda: gradcheck(lambda v: log(v[0] * v[0] + v[1]), [0.5, 2.0]))
check("log(exp(x)) == x, gradient 1", lambda: gradcheck(lambda v: log(exp(v[0])), [0.7]))

<details><summary><b>Solution</b> (try the exercise first!)</summary>The next cell is collapsed — click it to expand.</details>

In [ ]:
def log(v):
    assert v.data > 0, "log is only defined for positive values"
    out = Value(math.log(v.data), (v,), 'log')
    def _backward():
        v.grad += (1 / v.data) * out.grad
    out._backward = _backward
    return out

check("log forward", lambda: abs(log(Value(math.e)).data - 1.0) < 1e-12)
check("log backward", lambda: gradcheck(lambda v: log(v[0] * v[0] + v[1]), [0.5, 2.0]))
check("log(exp(x)) == x, gradient 1", lambda: gradcheck(lambda v: log(exp(v[0])), [0.7]))

### Exercise 2.3 — `tanh`
$\frac{d}{dx} \tanh x = 1 - \tanh^2 x$. Use `math.tanh` for the forward pass.

In [ ]:
def tanh(v):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
check("tanh forward", lambda: abs(tanh(Value(0.5)).data - math.tanh(0.5)) < 1e-12)
check("tanh backward", lambda: gradcheck(lambda v: tanh(v[0] * 3 + v[1]), [0.2, -0.4]))

<details><summary><b>Solution</b> (try the exercise first!)</summary>The next cell is collapsed — click it to expand.</details>

In [ ]:
def tanh(v):
    t = math.tanh(v.data)
    out = Value(t, (v,), 'tanh')
    def _backward():
        v.grad += (1 - t**2) * out.grad
    out._backward = _backward
    return out

check("tanh forward", lambda: abs(tanh(Value(0.5)).data - math.tanh(0.5)) < 1e-12)
check("tanh backward", lambda: gradcheck(lambda v: tanh(v[0] * 3 + v[1]), [0.2, -0.4]))

### Exercise 2.4 — `sigmoid`, two ways

$\sigma(x) = \frac{1}{1 + e^{-x}}$, and $\sigma'(x) = \sigma(x)(1 - \sigma(x))$.

1. `sigmoid_composed(v)`: build it **only out of existing operations** (`exp`, `+`, `/`, unary `-`). You write no backward pass
   at all — autograd derives it for you by chaining the pieces!
2. `sigmoid_fused(v)`: a single new op with its own hand-written `_backward`.

Both must pass gradcheck. Then compare how many graph nodes each one creates.

In [ ]:
def sigmoid_composed(v):
    # YOUR CODE HERE
    raise NotImplementedError

def sigmoid_fused(v):
    # YOUR CODE HERE
    raise NotImplementedError

In [ ]:
check("sigmoid_composed", lambda: gradcheck(lambda v: sigmoid_composed(v[0]), [0.3]))
check("sigmoid_fused", lambda: gradcheck(lambda v: sigmoid_fused(v[0]), [0.3]))
check("both agree", lambda: abs(sigmoid_composed(Value(0.3)).data - sigmoid_fused(Value(0.3)).data) < 1e-12)

<details><summary><b>Solution</b> (try the exercise first!)</summary>The next cell is collapsed — click it to expand.</details>

In [ ]:
def sigmoid_composed(v):
    return 1 / (1 + exp(-v))

def sigmoid_fused(v):
    s = 1 / (1 + math.exp(-v.data))
    out = Value(s, (v,), 'sigmoid')
    def _backward():
        v.grad += s * (1 - s) * out.grad
    out._backward = _backward
    return out

check("sigmoid_composed", lambda: gradcheck(lambda v: sigmoid_composed(v[0]), [0.3]))
check("sigmoid_fused", lambda: gradcheck(lambda v: sigmoid_fused(v[0]), [0.3]))
check("both agree", lambda: abs(sigmoid_composed(Value(0.3)).data - sigmoid_fused(Value(0.3)).data) < 1e-12)

In [ ]:
from nanograd.diagnostics import count_nodes
x = Value(0.3)
print("composed sigmoid nodes:", count_nodes(sigmoid_composed(x)))
print("fused sigmoid nodes:   ", count_nodes(sigmoid_fused(x)))

**Takeaway:** composing existing ops is always *correct* for free, but builds a bigger graph (more memory, more Python calls).
Real frameworks fuse common patterns like sigmoid, softmax and layer-norm into single ops for speed — and then rely on
gradient checks exactly like yours to make sure the hand-written backward is right.

## Part 3 — Break it on purpose: why `+=`?

Every `_backward` in the engine *adds* to `.grad` instead of assigning it. Let's see what goes wrong if we "simplify" `__add__`
to use `=`.

In [ ]:
original_add = Value.__add__

def broken_add(self, other):
    other = other if isinstance(other, Value) else Value(other)
    out = Value(self.data + other.data, (self, other), '+')
    def _backward():
        self.grad = out.grad   # BUG: = instead of +=
        other.grad = out.grad
    out._backward = _backward
    return out

Value.__add__ = broken_add

a = Value(3.0)
b = a + a          # db/da should be 2
b.backward()
print("a.grad with broken add:", a.grad)

With `a + a`, the same node `a` is **both** inputs. Its gradient must collect a contribution from each path through the graph
(this is the *multivariate chain rule*: sum over all paths). With `=`, the second write overwrites the first.

### Exercise 3.1 — Predict before you run
With the broken `__add__` still installed, which of these expressions would *still* get correct gradients?
Fill in `True` (still correct) or `False` (now wrong) for each, **then** run the check.

In [ ]:
predictions = {
    "a + b":         None,
    "a * b + a":     None,
    "(a + b) * b":   None,
    "a * a":         None,   # careful: this one uses __mul__, not __add__
    "(a + 1) + (a + 2)": None,
}
# YOUR ANSWERS ABOVE

In [ ]:
def actual(expr):
    return lib_gradcheck(lambda v: eval(expr, {}, {"a": v[0], "b": v[1]}), [1.5, -0.5])

def check_predictions():
    if any(p is None for p in predictions.values()):
        raise NotImplementedError
    for expr, guess in predictions.items():
        print(f"  {expr:<20} you said {str(guess):<5}  actual {actual(expr)}")
    return all(guess == actual(expr) for expr, guess in predictions.items())

check("predictions", check_predictions)

<details><summary><b>Solution</b> (try the exercise first!)</summary>The next cell is collapsed — click it to expand.</details>

In [ ]:
predictions = {
    "a + b":         True,   # each input is used once: overwriting a zero grad is the same as adding
    "a * b + a":     True,   # surprise! two paths into `a`, but the broken add runs FIRST (it is nearer the output),
                             # writing a.grad = 1; the multiply's backward then does a.grad += b on top of it
    "(a + b) * b":   False,  # `b` gets a gradient from the add and from the multiply; the add overwrites
    "a * a":         True,   # __mul__ still uses +=, and no add is involved
    "(a + 1) + (a + 2)": False,  # two paths into `a`, both through broken adds
}
check("predictions", check_predictions)

Notice that the error depends on the **order** in which `_backward` runs: whichever write happens last wins. Bugs like this
are silent — the network still trains, just badly — which is exactly why we gradient-check every op.

Restore the real `__add__` before moving on:

In [ ]:
Value.__add__ = original_add
a = Value(3.0); b = a + a; b.backward()
print("a.grad with the real add:", a.grad)

## Part 4 — Where gradient checks lie: kinks

`relu` has no derivative at exactly 0 — the slope is 0 on the left and 1 on the right. Watch what happens:

In [ ]:
for x0 in [-0.5, 0.5, 0.0]:
    v = Value(x0)
    v.relu().backward()
    numeric = numerical_grad(lambda v: v[0].relu(), [x0])[0]
    ok = lib_gradcheck(lambda v: v[0].relu(), [x0])
    print(f"relu at x={x0:+.1f}: analytic {v.grad}, numeric {numeric:.2f} -> {'PASS' if ok else 'FAIL'}")

At the kink the central difference averages the two slopes (0.5), while backprop picks one side (0). Neither is "wrong" —
the derivative simply doesn't exist there. In practice: **gradient-check at random points, away from kinks**.

## Summary
* Numerical gradients are slow but need no calculus — the perfect referee for backprop.
* Each op only needs its *local* derivative; the chain rule does the rest.
* Gradients **accumulate** (`+=`) because a value can influence the output along several paths.
* `exp`, `log`, `tanh` are now part of `nanograd.engine.Value` — in the next lesson we use `exp` and `log` to build softmax and cross-entropy.